In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt

# i am god

np.random.seed(42)
TRUE_SLOPE = 0.5       # MPa per meter of depth
TRUE_INTERCEPT = 20.0  # MPa at surface
TRUE_SIGMA = 4.0       # Measurement noise

n = 50
depth = np.random.uniform(10, 100, n)

ucs = TRUE_INTERCEPT + TRUE_SLOPE * depth + np.random.normal(0, TRUE_SIGMA, n)

plt.scatter(depth, ucs, alpha=0.6)
plt.xlabel("Depth (m)")
plt.ylabel("UCS (MPa)")
plt.title("Rock Strength vs Depth")
plt.tight_layout()
plt.show()

In [2]:
# simple model

with pm.Model() as regression_model:

    # prior: how does the ucs vary with depth; weakly informative, we expect positive (overburden)

    slope = pm.HalfNormal("slope", sigma=1)
    intercept=pm.Lognormal("intercept", mu = np.log(25), sigma=0.55)
    # for mu we can cuz its addititive -> multiplicative; center shifting
# sigma!= np.log(15) as it acts as an exponential multiplier
# 0.1 is tight, 0.3 is moderate, and 0.5 to 1.0 is extremely wide/uncertain.
# e^sigma times differ from mu

    sigma = pm.HalfNormal("sigma", sigma = 0.5)
    # value in mind, depending on distro we use on y



    exp_ucs = pm.Deterministic("exp_ucs", intercept+slope*depth)
    # what happens if i dont specify pm.det and only say mu = ... ?

    y = pm.Lognormal("y", mu=pm.math.log(exp_ucs), sigma=sigma, observed=ucs)

    # pm.math to keep with the blueprint vibes, np demands numbers RN


In [ ]:
# perform prior predictive checks

with regression_model:
    prior_checks = pm.sample_prior_predictive(samples=1000, random_seed=42)
    fig, ax = plt.subplots(figsize=(12, 6))
az.plot_ppc(prior_checks,
group="prior",
kind="kde",
ax=ax,
colors=['gray', 'black', 'blue'], alpha=0.8) # [Prior, Mean, Observed]


az.plot_kde(ucs, ax=ax, plot_kwargs={"color": "red", "linewidth": 3, "linestyle": "--"}, label="Actual Observed Data")

ax.set_title("Prior Predictive Check: Physicality Audit", fontsize=14, fontweight='bold')
ax.set_xlabel("UCS", fontsize=12)
ax.set_ylabel("Probability Density", fontsize=12)


ax.set_xlim(-10, max(ucs) * 2.5)

ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x_plot = np.linspace(0, 120, 100)

# Extract prior samples for slope and intercept
prior_slopes = prior_checks.prior["slope"].values.flatten()
prior_intercepts = prior_checks.prior["intercept"].values.flatten()


for i in range(50):
    y_line = prior_intercepts[i] + prior_slopes[i] * x_plot
    ax.plot(x_plot, y_line, 'gray', alpha=0.1)

ax.scatter(depth, ucs, c='blue', zorder=5, label='Actual data')

ax.set_xlabel("Depth (m)")
ax.set_ylabel("Expected UCS (MPa)")
ax.set_title("Prior Predictive: Do my prior LINES look reasonable?")

# Constrain the Y-axis so extreme priors don't ruin the scale
ax.set_ylim(-10, max(ucs) * 2)

ax.legend()
plt.tight_layout()
plt.show()